In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from openai import OpenAI
import os

In [4]:
# Инициализируем клиент OpenAI, но жестко перенаправляем его на официальный шлюз Google
client = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=os.environ.get("GEMINI_API_KEY")
)



In [5]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [6]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [7]:
print("lesson pages:", len(documents))

lesson pages: 72


In [8]:
from minsearch import Index

index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)
index.fit(documents)

In [9]:
index.search('How does the agentic loop keep calling the model until it stops?', num_results=1)

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

In [10]:
# Переписываем класс под нашу новую структуру документов (filename и content)
class GeminiRAG:
    def __init__(self, index, llm_client):
        self.index = index
        self.llm_client = llm_client
        self.instructions = (
            "Your task is to answer questions from the course participants based on the provided context. "
            "Use the context to find relevant information and provide accurate answers. "
            "If the answer is not found in the context, respond with 'I don't know.'"
        )

    def search(self, query, num_results=5):
        # Адаптируем поиск: ищем по полю 'content' в нашем индексе minsearch
        return self.index.search(
            query,
            filter_dict={}, # Убираем фильтр по курсу, так как в гитхабе его нет
            boost_dict={'content': 3.0}, # Даем максимальный вес тексту статьи
            num_results=num_results
        )

    def build_context(self, search_results):
        lines = []
        # Собираем контекст из новых полей: filename и content
        for doc in search_results:
            lines.append(f"File: {doc['filename']}")
            lines.append(f"Content: {doc['content']}")
            lines.append('')
        return '\n'.join(lines).strip()

    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return f"QUESTION: {query}\n\nCONTEXT:\n{context}"

    def llm(self, prompt):
        # Переписываем под рабочий синтаксис OpenAI SDK
        response = self.llm_client.chat.completions.create(
            model="gemini-2.5-flash",
            messages=[
                {'role': 'system', 'content': self.instructions},
                {'role': 'user', 'content': prompt}
            ]
        )
        return response.choices[0].message.content

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        return self.llm(prompt)

# Инициализируем систему (передаём сюда объект индекса minsearch)
rag_assistant = GeminiRAG(index=index, llm_client=client)

# Запускаем финальный вопрос
query = "How does the agentic loop keep calling the model until it stops?"
print(rag_assistant.rag(query))


The agentic loop keeps calling the model until it stops by wrapping the interaction in a `while` loop that continues as long as the model requests a function call.

Here's how it works:
1.  **Initial Call**: The loop starts by sending the user's question and initial instructions to the Language Model (LLM) along with the available tools (e.g., `search`).
2.  **Process Response**: The LLM responds. This response is examined:
    *   If the response contains a `function_call` (e.g., `search`), the associated function is executed with the arguments provided by the LLM. The output of this function call is then added to the conversation history. A flag, `has_function_calls`, is set to `True`.
    *   If the response is a `message` (the LLM's answer) and does not contain any function calls, the `has_function_calls` flag remains `False`.
3.  **Memory Update**: All parts of the LLM's response and any tool outputs are appended to the `messages` (conversation history). This history is crucial be

In [11]:
response = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {'role': 'system', 'content': rag_assistant.instructions},
        {'role': 'user', 'content': rag_assistant.build_prompt(query, rag_assistant.search(query))}
    ]
)
print(response.usage.prompt_tokens)
print(response.usage.completion_tokens)

7952
491
